<a href="https://colab.research.google.com/github/lauandsalih/Smart-Grid-Battery-Dispatch/blob/main/battery_dispatch_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import cvxpy as cp

def generate_synthetic_prices(hours=24):
    """
    Simulates a 24-hour electricity market price curve (e.g., Nord Pool).
    Prices peak in the morning and evening, and drop at night/midday.
    Values are in €/MWh.
    """
    np.random.seed(42)
    time = np.arange(hours)
    # Base pattern: high morning (hr 8), high evening (hr 19), low night (hr 3)
    base_price = 50 + 40 * np.sin(np.pi * (time - 6) / 12) + 30 * np.sin(np.pi * (time - 16) / 8)
    noise = np.random.normal(0, 10, hours)

    # Ensure no massive negative prices for this simple simulation, though they happen in reality!
    prices = np.maximum(base_price + noise, 5.0)
    return prices

def optimize_battery_dispatch(prices, capacity_mwh=10.0, max_power_mw=2.5, efficiency=0.9):
    """
    Optimizes the charge/discharge schedule of a battery to maximize profit.
    """
    T = len(prices)

    # 1. Define Optimization Variables
    p_charge = cp.Variable(T, nonneg=True)     # Power charged into battery (MW)
    p_discharge = cp.Variable(T, nonneg=True)  # Power discharged to grid (MW)
    energy_level = cp.Variable(T + 1, nonneg=True) # State of Charge (MWh)

    # 2. Define Constraints
    constraints = []

    # Power limits
    constraints += [p_charge <= max_power_mw]
    constraints += [p_discharge <= max_power_mw]

    # Energy capacity limits
    constraints += [energy_level <= capacity_mwh]
    constraints += [energy_level >= 0]

    # Initial and final battery state (start empty, end empty)
    constraints += [energy_level[0] == 0]
    constraints += [energy_level[T] == 0]

    # Energy balance dynamics (Accounting for round-trip efficiency)
    for t in range(T):
        constraints += [
            energy_level[t + 1] == energy_level[t] + (p_charge[t] * efficiency) - (p_discharge[t] / efficiency)
        ]

    # 3. Define Objective Function (Maximize Profit)
    # Profit = Revenue from discharging - Cost of charging
    profit = cp.sum(cp.multiply(p_discharge, prices) - cp.multiply(p_charge, prices))
    objective = cp.Maximize(profit)

    # 4. Solve the Linear Program
    problem = cp.Problem(objective, constraints)
    problem.solve()

    # 5. Extract and Format Results
    results = pd.DataFrame({
        'Hour': np.arange(T),
        'Price_EUR_MWh': prices,
        'Charge_MW': p_charge.value,
        'Discharge_MW': p_discharge.value,
        'State_of_Charge_MWh': energy_level.value[:-1]
    })

    # Clean up small floating point artifacts from the solver
    results = results.round(2)

    print(f"Optimization Status: {problem.status}")
    print(f"Total Optimal Profit: €{problem.value:.2f}\n")

    return results

if __name__ == "__main__":
    print("Fetching day-ahead market prices...")
    market_prices = generate_synthetic_prices(24)

    print("Running Linear Programming Optimizer...\n")
    dispatch_schedule = optimize_battery_dispatch(
        prices=market_prices,
        capacity_mwh=10.0,  # 10 MWh battery
        max_power_mw=2.5,   # Can charge/discharge fully in 4 hours
        efficiency=0.9      # 90% round-trip efficiency
    )

    print("--- Optimal 24-Hour Battery Dispatch Schedule ---")
    print(dispatch_schedule.to_string(index=False))

Fetching day-ahead market prices...
Running Linear Programming Optimizer...

Optimization Status: optimal
Total Optimal Profit: €388.45

--- Optimal 24-Hour Battery Dispatch Schedule ---
 Hour  Price_EUR_MWh  Charge_MW  Discharge_MW  State_of_Charge_MWh
    0          14.97       2.50           0.0                 0.00
    1          21.46       2.50           0.0                 2.25
    2          43.05       2.50           0.0                 4.50
    3          64.66       0.00           0.0                 6.75
    4          57.66       1.76           0.0                 6.75
    5          65.02       0.00           0.0                 8.33
    6          87.01       0.00           2.5                 8.33
    7          79.51       0.00           2.5                 5.56
    8          65.31       0.00           0.0                 2.78
    9          72.23       0.00           2.5                 2.78
   10          58.79       0.00           0.0                 0.00
   11    